# LinearRegression from Scratch


# 구현할 것
- 공부시간과 성적간의 관계를 모델링한다.
    - **머신러닝 모델(모형)이란** 수집한 데이터를 기반으로 입력값(Feature)와 출력값(Target)간의 관계를 하나의 공식으로 정의한 함수이다. 그 공식을 찾는 과정을 **모델링**이라고 한다.
    - 이 예제에서는 공부한 시험시간으로 점수를 예측하는 모델을 정의한다.
    - 입력값과 출력값 간의 관계를 정의할 수있는 다양한 함수(공식)이 있다. 여기에서는 딥러닝과 관계가 있는 **Linear Regression** 을 사용해본다.

# 데이터 확인
- 입력데이터: 공부시간
- 출력데이터: 성적

|공부시간|점수|
|-|-|
|1|20|
|2|40|
|3|60|

우리가 수집한 공부시간과 점수 데이터를 바탕으로 둘 간의 관계를 식으로 정의 할 수 있으면 **내가 몇시간 공부하면 점수를 얼마 받을 수 있는지 예측할 수 있게 된다.**   
수집한 데이터를 기반으로 앞으로 예측할 수있는 모형을 만드는 것이 머신러닝 모델링이다.

  

## 학습(훈련) 데이터셋 만들기
- 모델을 학습시키기 위한 데이터셋을 구성한다.
- 입력데이터와 출력데이터을 각각 다른 행렬로 구성한다.
- 하나의 데이터 포인트의 입력/출력 값은 같은 index에 정의한다.

### 선형회귀 (Linear Regression)
- Feature들의 가중합을 이용해 Target을 추정한다.
- Feature에 곱해지는 가중치(weight)들은 각 Feature가 Target 얼마나 영향을 주는지 영향도가 된다.
    - 음수일 경우는 target값을 줄이고 양수일 경우는 target값을 늘린다.
    - 가중치가 0에 가까울 수록 target에 영향을 주지 않는 feature이고 0에서 멀수록 target에 많은 영향을 준다.
- 모델 학습과정에서 가장 적절한 Feature의 가중치를 찾아야 한다.
      

\begin{align}
&\large \hat{y} = W\cdot X + b\\
&\small \hat{y}: \text{모델추정값}\\
&\small W: \text{가중치}\\
&\small X: \text{Feature(입력값)}\\
&\small b: \text{bias(편향)}
\end{align}



## Train dataset 구성
- Train data는 feature(input)와 target(output) 각각 2개의 행렬로 구성한다.
- Feature의 행은 관측치(개별 데이터)를 열을 Feature(특성, 변수)를 표현한다. 이 문제에서는 `공부시간` 1개의 변수를 가진다.
- Target은 모델이 예측할 대상으로 행은 개별 관측치, 열은 각 항목에 대한 정답으로 구성한다.   
  이 문제에서 예측할 항목은 `시험점수` 한개이다.

In [ ]:
import torch

X_train = torch.tensor([[1.0], [2.0], [3.0]])
y_train = torch.tensor([[20.], [40.], [60.]])

X_train.size(), y_train.size()

(torch.Size([3, 1]), torch.Size([3, 1]))

## 파라미터 (weight, bias) 정의
- 학습대상/최적화 대상

In [ ]:
# weight: 형태 - feature가 1개 (1, 출력값의 개수-1)
weight = torch.randn(1, 1, requires_grad=True)   # [1: feature수, 1: 정답개수]
bias = torch.randn(1, requires_grad=True)  #  1: 정답 개수

print(weight.size(), bias.size())

torch.Size([1, 1]) torch.Size([1])


In [ ]:
weight

tensor([[0.1844]], requires_grad=True)

In [ ]:
X_train.dtype, weight.dtype, bias.dtype

(torch.float32, torch.float32, torch.float32)

In [ ]:
bias

tensor([0.4226], requires_grad=True)

In [ ]:
#### 추론
pred = X_train @ weight + bias
print("추론한 시험점수:", pred)

추론한 시험점수: tensor([[0.2890],
        [0.4734],
        [0.6579]], grad_fn=<AddBackward0>)


In [ ]:
#### 오차계산 (MSE)
loss = torch.mean((pred - y_train)**2, dim=0)
loss

tensor([1824.1217], grad_fn=<MeanBackward1>)

In [ ]:
##### weight와 bais를 업데이트
## 1. gradient 를 계산 (loss / weight, loss / bias)
loss.backward()

In [ ]:
print(weight.data, bias.data)
print(weight.grad, bias.grad)

tensor([[0.1844]]) tensor([0.1045])
tensor([[-184.5271]]) tensor([-79.0532])


In [ ]:
#### 2. weight, bias 를 업데이트
###  현재값 - 학습률 * grad
lr = 0.1
weight.data = weight.data - lr * weight.grad
bias.data = bias.data - lr * bias.grad

print("새 weight, bias")
print(weight.data, bias.data)

새 weight, bias
tensor([[18.6372]]) tensor([8.0098])


In [ ]:
# 다시 추론
pred2 = X_train @ weight + bias
print(pred2)

tensor([[26.6470],
        [45.2841],
        [63.9213]], grad_fn=<AddBackward0>)


In [ ]:
pred

tensor([[0.2890],
        [0.4734],
        [0.6579]], grad_fn=<AddBackward0>)

In [ ]:
# 오차
torch.mean((pred2 - y_train)**2)

tensor(29.1605, grad_fn=<MeanBackward0>)

In [ ]:
loss

tensor([1824.1217], grad_fn=<MeanBackward1>)

### 모델링

In [ ]:
#  모델 정의(함수 또는 클래스로 정의)
weigth = torch.randn(1, 1, requires_grad=True)
bias = torch.randn(1, requires_grad=True)

def linear_model(X):
    return X @ weight + bias

In [ ]:
# 오차계산 함수(MSE)
def mse_loss(pred, y):
    return torch.mean((pred - y) ** 2)

### 학습
1. 모델을 이용해 추정한다.
   - pred = model(input)
1. loss를 계산한다.
   - loss = loss_fn(pred, target)
1. 계산된 loss를 파라미터에 대해 미분하여 계산한 gradient 값을 각 파라미터에 저장한다.
   - loss.backward()
1. optimizer를 이용해 파라미터를 update한다.
   - optimizer.step()  
1. 파라미터의 gradient(미분값)을 0으로 초기화한다.
   - optimizer.zero_grad()
- 위의 단계를 반복한다.   

In [ ]:
epochs = 2000  # 최적화작업을 몇번 반복할지
lr = 0.01

for epoch in range(epochs):
    # 1. 추론
    pred = linear_model(X_train)
    # 2. 오차 계산
    loss = mse_loss(pred,  y_train)
    # 3. 파라미터들에 대한 gradient 계산.
    loss.backward()
    # 4. 파라미터 업데이트 (최적화)
    weight.data = weight.data - lr * weight.grad
    bias.data = bias.data - lr * bias.grad
    # 5. 파라미터 grad 초기화
    weight.grad = None
    bias.grad = None

    if epoch % 100 == 0 or epoch == (epochs-1): # 100번 반복당, 마지막에 loss출력 (로그)
        print(f"[{epoch+1}/{epochs}] - Loss : {loss.item()}")


[1/2000] - Loss : 9.165179252624512
[101/2000] - Loss : 0.005749539937824011
[201/2000] - Loss : 0.003553178394213319
[301/2000] - Loss : 0.0021955999545753
[401/2000] - Loss : 0.0013567848363891244
[501/2000] - Loss : 0.000838512263726443
[601/2000] - Loss : 0.0005181956803426147
[701/2000] - Loss : 0.00032030013971962035
[801/2000] - Loss : 0.00019802099268417805
[901/2000] - Loss : 0.0001224010338773951
[1001/2000] - Loss : 7.56988738430664e-05
[1101/2000] - Loss : 4.684828672907315e-05
[1201/2000] - Loss : 2.8977045076317154e-05
[1301/2000] - Loss : 1.7941370970220305e-05
[1401/2000] - Loss : 1.1107505997642875e-05
[1501/2000] - Loss : 6.8785398070758674e-06
[1601/2000] - Loss : 4.273383183317492e-06
[1701/2000] - Loss : 2.6392092422611313e-06
[1801/2000] - Loss : 1.6466461829622858e-06
[1901/2000] - Loss : 1.0126932465936989e-06
[2000/2000] - Loss : 6.356870585477736e-07


In [ ]:
weight.data, bias.data

(tensor([[20.0009]]), tensor([-0.0021]))

In [ ]:
with torch.no_grad():
    p = linear_model(X_train)
    print(p)

tensor([[19.9988],
        [39.9998],
        [60.0007]])


In [ ]:
time = torch.tensor([[6], [8.2]], dtype=torch.float32)
with torch.no_grad():
    p2 = linear_model(time)
    print(p2)


tensor([[120.0035],
        [164.0055]])


# 다중 입력, 다중 출력
- 다중입력: Feature가 여러개인 경우
- 다중출력: Output 결과가 여러개인 경우

다음 가상 데이터를 이용해 사과와 오렌지 수확량을 예측하는 선형회귀 모델을 정의한다.  
[참조](https://www.kaggle.com/code/aakashns/pytorch-basics-linear-regression-from-scratch)


|온도(F)|강수량(mm)|습도(%)|사과생산량(ton)|오렌지생산량|
|-|-|-|-:|-:|
|73|67|43|56|70|
|91|88|64|81|101|
|87|134|58|119|133|
|102|43|37|22|37|
|69|96|70|103|119|

```
사과수확량  = w11 * 온도 + w12 * 강수량 + w13 * 습도 + b1
오렌지수확량 = w21 * 온도 + w22 * 강수량 + w23 *습도 + b2
```

- `온도`, `강수량`, `습도` 값이 **사과**와, **오렌지 수확량**에 어느정도 영향을 주는지 가중치를 찾는다.
    - 모델은 사과의 수확량, 오렌지의 수확량 **두개의 예측결과를 출력**해야 한다.
    - 사과에 대해 예측하기 위한 weight 3개와 오렌지에 대해 예측하기 위한 weight 3개 이렇게 두 묶음, 총 6개의 weight를 정의하고 학습을 통해 가장 적당한 값을 찾는다.
        - `개별 과일를 예측하기 위한 weight들 @ feature들` 의 계산 결과를  **Node, Unit, Neuron** 이라고 한다.
        - 두 과일에 대한 Unit들을 묶어서 **Layer** 라고 한다.
- 목적은 우리가 수집한 train 데이터셋을 이용해 **정확한 예측을 위한 weight와 bias 들**을 찾는 것이다.

## Train Dataset
- Train data는 feature(input)와 target(output) 각각 2개의 행렬로 구성한다.
- Feature의 행은 관측치(개별 데이터)를 열을 Feature(특성, 변수)를 표현한다. 이 문제에서는 `온도, 강수량, 습도` 세개의 변수를 가진다.
- Target은 모델이 예측할 대상으로 행은 개별 관측치, 열은 각 항목에 대한 정답으로 구성한다. 이 문제에서 예측할 항목은 `사과수확량, 오렌지 수확량` 2개의 값이다.

In [1]:
#  input: 생산환경 (temp, rainfall, humidity) : (5, 3)
environs = [
    [73, 67, 43],
    [91, 88, 64],
    [87, 134, 58],
    [102, 43, 37],
    [69, 96, 70]
]

# Targets: 생산량 - (apples, oranges) - (5, 2)
apple_orange_output = [
    [56, 70],
    [81, 101],
    [119, 133],
    [22, 37],
    [103, 119]
]

In [2]:
import torch
# Dataset을 torch.Tensor로 생성
X = torch.tensor(environs, dtype=torch.float32)
y = torch.tensor(apple_orange_output, dtype=torch.float32)
X.shape, y.shape

(torch.Size([5, 3]), torch.Size([5, 2]))

In [3]:
X

tensor([[ 73.,  67.,  43.],
        [ 91.,  88.,  64.],
        [ 87., 134.,  58.],
        [102.,  43.,  37.],
        [ 69.,  96.,  70.]])

## weight와 bias
- weight: 각 feature들이 생산량에 영향을 주었는지의 가중치로 feature에 곱해줄 값.
    - 사과, 오렌지의 생산량을 구해야 하므로 가중치가 두개가 된다.
    - weight의 shape: `(3, 2)`
- bias는 모든 feature들이 0일때 생산량이 얼마일지를 나타내는 값으로 feature와 weight간의 가중합 결과에 더해줄 값이다.
    - 사과, 오렌지의 생산량을 구하므로 bias가 두개가 된다.
    - bias의 shape: `(2, )`

### Linear Regression model
모델은 weights `w`와 inputs `x`의 내적(dot product)한 값에 bias `b`를 더하는 함수.

$$
\hspace{2.5cm} X \hspace{1.1cm} \cdot \hspace{1.2cm} W \hspace{1.2cm}  + \hspace{1cm} b \hspace{2cm}
$$

$$
\left[ \begin{array}{cc}
73 & 67 & 43 \\
91 & 88 & 64 \\
\vdots & \vdots & \vdots \\
69 & 96 & 70
\end{array} \right]
%
\cdot
%
\left[ \begin{array}{cc}
w_{11} & w_{21} \\
w_{12} & w_{22} \\
w_{13} & w_{23}
\end{array} \right]
%
+
%
\left[ \begin{array}{cc}
b_{1} & b_{2} \\
b_{1} & b_{2} \\
\vdots & \vdots \\
b_{1} & b_{2} \\
\end{array} \right]
$$


<center style="font-size:0.9em">
$w_{11},\,w_{12},\,w_{13}$: 사과 생산량 계산시 각 feature들(생산환경)에 곱할 가중치   <br>
$w_{21},\,w_{22},\,w_{23}$: 오렌지 생산량 계산시 각 feature들(생산환경)에 곱할 가중치    
</center>

<center>
<img src="https://raw.githubusercontent.com/kgmyhGit/image_resource/main/deeplearning/figures/3_unit_layer.png">
</center>

In [4]:
##### 모델링
# weight shape: (3: feature개수, 2: output의 개수)
weight = torch.randn(3, 2, requires_grad=True)
# bias shape: (2: output의 개수, )
bias = torch.randn(2, requires_grad=True)

def model(X):
    return X @ weight + bias

In [6]:
y_hat = model(X)
print(y_hat.shape)  # (5: 개수, 2: 과일생산량(사과, 오렌지))
y_hat

torch.Size([5, 2])


tensor([[-259.9341,   36.1067],
        [-328.7338,   29.9280],
        [-415.0042,   64.9064],
        [-270.2454,   57.3424],
        [-300.0131,    5.0643]], grad_fn=<AddBackward0>)

In [8]:
y

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [10]:
#  Loss 함수
def loss_fn(pred, y):
    """mean squared error"""
    return torch.mean((pred - y)**2)

In [11]:
## 학습 (trainset을 이용해서 weight, bias 를 최적화)
epochs = 5000
lr = 0.00001

for epoch in range(epochs):
    # 1. 모델을 이용해서 예측
    pred = model(X)
    # 2. 오차 계산
    loss = loss_fn(pred, y)
    # 3. weight와 bias의 gradient를 계산
    loss.backward()
    # 4. weight와 bias 최적화
    weight.data = weight.data - lr * weight.grad
    bias.data = bias.data - lr * bias.grad
    # 5. grad 초기화
    weight.grad = None
    bias.grad = None

    # 로그 출력 (epoch별 loss 출력)
    ### 100 epoch당 1번씩, 마지막 epoch
    if epoch % 100 == 0 or epoch == (epochs - 1):
        print(f"[{epoch}/{epochs}] - loss: {loss.item(): .5f}")

[0/5000] - loss:  82491.54688
[100/5000] - loss:  474.57608
[200/5000] - loss:  295.08432
[300/5000] - loss:  217.19760
[400/5000] - loss:  172.51080
[500/5000] - loss:  141.02986
[600/5000] - loss:  116.48206
[700/5000] - loss:  96.55667
[800/5000] - loss:  80.14979
[900/5000] - loss:  66.57271
[1000/5000] - loss:  55.31871
[1100/5000] - loss:  45.98482
[1200/5000] - loss:  38.24192
[1300/5000] - loss:  31.81840
[1400/5000] - loss:  26.48936
[1500/5000] - loss:  22.06826
[1600/5000] - loss:  18.40039
[1700/5000] - loss:  15.35742
[1800/5000] - loss:  12.83291
[1900/5000] - loss:  10.73852
[2000/5000] - loss:  9.00099
[2100/5000] - loss:  7.55942
[2200/5000] - loss:  6.36349
[2300/5000] - loss:  5.37133
[2400/5000] - loss:  4.54820
[2500/5000] - loss:  3.86531
[2600/5000] - loss:  3.29876
[2700/5000] - loss:  2.82875
[2800/5000] - loss:  2.43880
[2900/5000] - loss:  2.11529
[3000/5000] - loss:  1.84691
[3100/5000] - loss:  1.62424
[3200/5000] - loss:  1.43952
[3300/5000] - loss:  1.286

In [12]:
print(weight)
print("---------------")
print(bias)

tensor([[-0.3912, -0.3087],
        [ 0.8442,  0.8032],
        [ 0.7059,  0.8790]], requires_grad=True)
---------------
tensor([-1.4425,  1.5024], requires_grad=True)


In [13]:
#### 학습된 모델로 추론
with torch.no_grad():
    p = model(X)
    print(p)

tensor([[ 56.9155,  70.5799],
        [ 82.4263, 100.3498],
        [118.5909, 133.2564],
        [ 21.0724,  37.0785],
        [102.0229, 118.8398]])


In [14]:
y

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [19]:
new_X = [[63.2, 121.6, 32.1]] #, [63.2, 121.6, 32.1], [63.2, 121.6, 32.1]]
new_X = torch.tensor(new_X, dtype=torch.float32)
with torch.no_grad():
    new_pred = model(new_X)
    print(new_pred)

print("사과 예측 생산량:", new_pred[0, 0].item())
print("오렌지 예측 생산량:", new_pred[0, 1].item())

tensor([[ 99.1507, 107.8770],
        [ 99.1507, 107.8770],
        [ 99.1507, 107.8770]])
사과 예측 생산량: 99.1506576538086
오렌지 예측 생산량: 107.876953125


# pytorch built-in 모델을 사용해 Linear Regression 구현

In [20]:
inputs = torch.tensor(
    [[73, 67, 43],
     [91, 88, 64],
     [87, 134, 58],
     [102, 43, 37],
     [69, 96, 70]], dtype=torch.float32)

In [21]:
targets = torch.tensor(
    [[56, 70],
    [81, 101],
    [119, 133],
    [22, 37],
    [103, 119]], dtype=torch.float32)

## torch.nn.Linear
Pytorch는 torch.nn.Linear 클래스를 통해 Linear Regression 모델을 제공한다.  
torch.nn.Linear에 입력 feature의 개수와 출력 값의 개수를 지정하면 random 값으로 초기화한 weight와 bias들을 생성해 모델을 구성한다.
- `torch.nn.Linear(input feature의 개수 , output 값의 개수)`

## Optimizer와 Loss 함수 정의
- **Optimizer**: 계산된 gradient값을 이용해 파라미터들을 업데이트 하는 함수
- **Loss 함수**: 정답과 모델이 예측한 값사이의 차이(오차)를 계산하는 함수.
  - 모델을 최적화하는 것은 이 함수의 값을 최소화하는 것을 말한다.
- `torch.optim` 모듈에 다양한 Optimizer 클래스가 구현되있다.
- `torch.nn` 또는 `torch.nn.functional` 모듈에 다양한 Loss 함수가 제공된다.

In [22]:
inputs.shape, targets.shape

(torch.Size([5, 3]), torch.Size([5, 2]))

In [24]:
# 선형회귀 모델을 정의. torch.nn.Linear 클래스
import torch
import torch.nn as nn

model = nn.Linear(3, 2)  # 3: input feature 개수, 2: output 수

In [25]:
# loss 함수
loss_fn = torch.nn.MSELoss()  # 클래스
# loss_fn = torch.nn.functional.mse_loss # 함수

In [26]:
# optimizer (torch.optim 모듈에 정의): weight.data = weight.data - lr * weight.grad
optimizer = torch.optim.SGD(
    model.parameters(), # 최적화 대상 파라미터들(weight, bias 들)을 model에서 조회해서 전달.
    lr = 0.00001,       # Learning Rage
)

In [28]:
list(model.parameters())

[Parameter containing:
 tensor([[ 0.5189, -0.1791, -0.4192],
         [-0.4757, -0.5289,  0.4623]], requires_grad=True),
 Parameter containing:
 tensor([0.3587, 0.5068], requires_grad=True)]

## Model Train

In [32]:
# model(inputs) # callable
class Test:

    def __call__(self, x, y):
        return x + y

t = Test()
t(10, 20)
# model

Linear(in_features=3, out_features=2, bias=True)

In [33]:
epochs = 5000

for epoch in range(epochs):
    # 추론
    pred = model(inputs)
    # loss 계산
    loss = loss_fn(pred, targets) # torch.nn.functional.mse_loss(pred, targets) # (모델추정값, 정답)
    # gradient 계산
    loss.backward()
    # 파라미터 업데이트: optimizer.step()
    optimizer.step() # w.data = w.data - lr * w.grad

    # 파라미터 초기화 w.grad=None, b.grad=None
    optimizer.zero_grad()

    # 현재 epoch 학습 결과를 log로 출력
    if epoch % 100 == 0 or epoch == epochs-1:
        print(f"[{epoch+1:04d}/{epochs}] - {loss.item()}")

[0001/5000] - 16051.3544921875
[0101/5000] - 267.88079833984375
[0201/5000] - 90.64125061035156
[0301/5000] - 38.3353157043457
[0401/5000] - 21.52486801147461
[0501/5000] - 15.038106918334961
[0601/5000] - 11.74980354309082
[0701/5000] - 9.609119415283203
[0801/5000] - 7.997957706451416
[0901/5000] - 6.707547664642334
[1001/5000] - 5.649981498718262
[1101/5000] - 4.776245594024658
[1201/5000] - 4.052388668060303
[1301/5000] - 3.4521470069885254
[1401/5000] - 2.9542555809020996
[1501/5000] - 2.541191339492798
[1601/5000] - 2.198539972305298
[1701/5000] - 1.9142544269561768
[1801/5000] - 1.678400993347168
[1901/5000] - 1.482729196548462
[2001/5000] - 1.3203933238983154
[2101/5000] - 1.1857136487960815
[2201/5000] - 1.0739959478378296
[2301/5000] - 0.9813024401664734
[2401/5000] - 0.9043985605239868
[2501/5000] - 0.840599536895752
[2601/5000] - 0.7876710295677185
[2701/5000] - 0.7437639236450195
[2801/5000] - 0.7073307633399963
[2901/5000] - 0.6771084666252136
[3001/5000] - 0.652029812335

In [34]:
# 추론 => gradient 계산을 할 필요가 없다. ==> grad_fn을 만들 필요가 없다. 그래서 torch.no_grad() 블록에서 추론 작업을 실행한다.
with torch.no_grad():
    pred = model(inputs)

In [35]:
targets

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [36]:
pred

tensor([[ 57.2313,  70.3935],
        [ 82.1161, 100.6238],
        [118.7943, 132.9299],
        [ 21.1050,  37.0004],
        [101.8192, 119.1662]])

In [38]:
loss_fn(pred, targets)

tensor(0.5328)

In [39]:
# 학습 로직을 함수 구현
def train(inputs, targets, epochs, model, loss_fn, optimizer):

    for epoch in range(epochs):
        # 추론
        pred = model(inputs)
        # loss 계산
        loss = loss_fn(pred, targets) # torch.nn.functional.mse_loss(pred, targets) # (모델추정값, 정답)
        # gradient 계산
        loss.backward()
        # 파라미터 업데이트: optimizer.step()
        optimizer.step()
        # 파라미터 초기화 w.grad=None, b.grad=None
        optimizer.zero_grad()
        # 현재 epoch 학습 결과를 log로 출력
        if epoch % 100 == 0 or epoch == epochs-1:
            print(f"[{epoch+1:04d}/{epochs}] - {loss.item()}")

In [40]:
model = nn.Linear(3, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

In [41]:
train(inputs, targets, 5000, model, nn.functional.mse_loss, optimizer)

[0001/5000] - 1941.406982421875
[0101/5000] - 5.849392414093018
[0201/5000] - 1.32497239112854
[0301/5000] - 0.6318123936653137
[0401/5000] - 0.5255816578865051
[0501/5000] - 0.5093026161193848
[0601/5000] - 0.5068023800849915
[0701/5000] - 0.5064173340797424
[0801/5000] - 0.5063559412956238
[0901/5000] - 0.506341278553009
[1001/5000] - 0.5063381195068359
[1101/5000] - 0.5063298940658569
[1201/5000] - 0.50632643699646
[1301/5000] - 0.5063239336013794
[1401/5000] - 0.5063199996948242
[1501/5000] - 0.5063170790672302
[1601/5000] - 0.5063163638114929
[1701/5000] - 0.5063132047653198
[1801/5000] - 0.5063067078590393
[1901/5000] - 0.5063031911849976
[2001/5000] - 0.5063010454177856
[2101/5000] - 0.5062928199768066
[2201/5000] - 0.5062925815582275
[2301/5000] - 0.5062892436981201
[2401/5000] - 0.5062847137451172
[2501/5000] - 0.5062812566757202
[2601/5000] - 0.5062763094902039
[2701/5000] - 0.5062748789787292
[2801/5000] - 0.5062676668167114
[2901/5000] - 0.5062629580497742
[3001/5000] - 0.5

In [43]:
with torch.no_grad():
    p = model(inputs)
    print(p)

tensor([[ 57.1053,  70.2720],
        [ 82.2447, 100.6936],
        [118.7023, 132.9640],
        [ 21.0889,  37.0198],
        [101.9113, 119.1324]])
